In [1]:
## Import libraries
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support,classification_report
import os
from sklearn.cluster import AgglomerativeClustering
import numpy as np

In [2]:
def compute_metrics_pi(df):
    # Filter the DataFrame
    df_filtered = df.query("dataset=='mrpc' or dataset=='how'").copy()
    df_filtered['prediction'] = (df_filtered.prediction > 0).astype(int)

    # Calculate precision, recall, F1 score, and support
    grouped_metrics = df_filtered.groupby(['dataset']).apply(lambda x: pd.Series(
        precision_recall_fscore_support(x['label'], x['prediction'], average='weighted')[:-1],  # Drop the 'support' column
        index=['Precision', 'Recall', 'F1 Score']
    ))
    
    return grouped_metrics


In [3]:
from utils import load_results_to_dict

In [6]:
dict_predictions = load_results_to_dict("results/")
# Open a Markdown file for writing
with open('results.md', 'w') as markdown_file:

    # Iterate through dictionary items
    for k, v in dict_predictions.items():
        # Write file name to Markdown
        markdown_file.write(f"## File: {k}\n")
        
        # Write metrics table to Markdown
        metrics = compute_metrics_pi(v)
        markdown_file.write(metrics.to_markdown() + '\n')


c:\ProgramData\miniconda3\envs\pgkpr\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
c:\ProgramData\miniconda3\envs\pgkpr\lib\site-packages\sklearn\metrics\_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


# Create clusters for NewsWCL50

In [4]:
def extract_mention_i(text,i):
    text_list = text.split("[SEP]")
    return text_list[i]

In [5]:
def predict_clusters(input_df):
    df_exp = input_df.copy()

    #Extract mentions id
    df_exp["mention1_id"] = df_exp.instance_id.apply(lambda x:extract_mention_i(x,1) )
    df_exp["mention2_id"] = df_exp.instance_id.apply(lambda x:extract_mention_i(x,2) )

    # 1. Turn logit into probability
    df_exp['predicted_prob'] = 1 / (1 + np.exp(-df_exp['prediction']))
    # 2. Compute distance
    df_exp['distance'] = 1-df_exp['predicted_prob'] 
    # 3. Map mentions to indices

    first_set = set(df_exp.mention1_id)
    second_set = set(df_exp.mention2_id)
    union_set = first_set.union(second_set)
    num_mentions = len(union_set)

    mention_index_dict = {value: idx for idx, value in enumerate(union_set)}
    index_mention_dict = {idx: value for idx, value in enumerate(union_set)}

    df_exp["mention1_id_index"] =  df_exp["mention1_id"].map(mention_index_dict)
    df_exp["mention2_id_index"] =  df_exp["mention2_id"].map(mention_index_dict)
    # Create square matrix of 0s (minimum distance) and replace combinations with actual distances
    distance_matrix = np.zeros((num_mentions,num_mentions))

    mention1_indices = df_exp["mention1_id_index"].values
    mention2_indices = df_exp["mention2_id_index"].values
    distances = df_exp["distance"].values

    distance_matrix[mention1_indices, mention2_indices] = distances
    distance_matrix[mention2_indices, mention1_indices] = distances

    #Name matrix with mention ids
    final_distance_matrix = pd.DataFrame(distance_matrix)

    column_names = []
    for col in final_distance_matrix.columns:
        column_names.append(index_mention_dict[col])

    final_distance_matrix.columns=column_names

    #Apply agglomerative clustering
    clustering = AgglomerativeClustering(n_clusters=None, metric='precomputed', linkage='average',
                                            distance_threshold=0.7)
    fit = clustering.fit(final_distance_matrix)
    return pd.DataFrame({'mention_id':fit.feature_names_in_,"predicted_cluster":fit.labels_})

In [10]:
dict_predicted_clusters = {}
for file_name, pairwise_predictions in dict_predictions.items():
    list_predicted_clusters_per_file = []
    #Choose one df
    chosen_df = pairwise_predictions.query("dataset=='NewsWCL50-prep'")
    unique_mention_types = chosen_df.mention_type.unique()
    dict_combine_topics = {}
    #new file key
    new_file_key = file_name.split(".")[0].replace('predicted_pairs','cluster')

    #Create all the pairs mention type, subtopic
    for mention_type in unique_mention_types:
        dict_combine_topics[mention_type] = chosen_df[chosen_df['mention_type']==mention_type].subtopic.unique()

    for entity_type,subtopics in dict_combine_topics.items():
        for subtopic in subtopics:
            chosen_df_subtopic = chosen_df[(chosen_df['mention_type'] == entity_type) & (chosen_df['subtopic'] == subtopic)]
            predicted_clusters_df = predict_clusters(chosen_df_subtopic)
            predicted_clusters_df = predicted_clusters_df.rename(columns={'predicted_cluster':new_file_key})
            list_predicted_clusters_per_file.append(predicted_clusters_df)
    
    dict_predicted_clusters[new_file_key] = pd.concat(list_predicted_clusters_per_file)


In [11]:
merged_df = list(dict_predicted_clusters.values())[0]
for key, df in list(dict_predicted_clusters.items())[1:]:
    merged_df = merged_df.merge(df, on='mention_id', how='outer')

In [12]:
merged_df

,mention_id,cluster_exp1_seq_sampling,cluster_exp1_sim_sampling,cluster_exp2_seq_sampling,cluster_exp2_sim_sampling,cluster_exp3_seq_sampling,cluster_exp3_sim_sampling,cluster_exp4_seq_sampling,cluster_exp4_sim_sampling
0,36_8ecb_LOC17642835073942721_0_3_Bagp,0,0,0,0,6,6,6,13
1,36_7ecb_HUM17642655532934542_1_9_hjnJ,1,16,0,0,2,14,23,12
2,36_1ecb_HUM18520763746825423_0_3_MKoY,0,8,0,0,12,9,9,2
3,36_5ecb_HUM17720489125766418_1_20_izrE,3,17,0,0,17,8,9,10
4,36_3ecb_LOC17642819306422986_1_8_AgDi,0,18,0,0,6,16,1,12
...,...,...,...,...,...,...,...,...,...
3818,45_3ecbplus_ACT18223452941764084_4_21_W56F,7,10,0,0,12,17,13,18
3819,45_8ecbplus_ACT18223282001061069_1_2_9U2r,4,0,0,0,3,2,8,4
3820,45_1ecbplus_ACT18223465632264326_3_10_UWpQ,0,18,0,0,1,1,2,22
3821,45_3ecbplus_ACT18223480382299084_3_4_HZnX,2,5,0,0,2,13,18,9


In [13]:
column_to_check = 'cluster_exp1_seq_sampling'
column_to_compare = 'cluster_exp4_sim_sampling'
are_values_equal = (merged_df[column_to_check] == merged_df[column_to_compare]).all()

if are_values_equal:
    print(f"All values in {column_to_check} are equal to all values in {column_to_compare}.")
else:
    print(f"Not all values in {column_to_check} are equal to all values in {column_to_compare}.")

Not all values in cluster_exp1_seq_sampling are equal to all values in cluster_exp4_sim_sampling.


In [14]:
merged_df.to_excel("final_clusters_ecb.xlsx",index=False)